# Matryoshka Finetuning

In [1]:
import sys, os, yaml
import pandas as pd
from pathlib import Path
import torch, pickle

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.esci.data import sample_dataset
from src.esci.matryoshka_train import train_matryoshka
from src.esci.system_a import encode_systemA
from src.esci.system_b import build_candidates
from src.esci.metrics import compute_recall_metrics, compute_ndcg_metrics
from src.esci.sparse_retrievers import SPLADEFast, BM25Fast

# Load Config
with open(project_root / "configs" / "esci.yaml") as f:
    cfg = yaml.safe_load(f)

# Load Data
print("Loading Processed Data...")
pair_df = pd.read_parquet(cfg["paths"]["processed_dir"] + "/pair_df.parquet")

# Build Ground Truth (Critical for Metrics)
print("📊 Building Ground Truth Map (q_rels)...")
q_rels = pair_df.groupby("query_id")["grade"].apply(list).to_dict()

Loading Processed Data...
📊 Building Ground Truth Map (q_rels)...


### Training Execution

In [5]:
# TRAINING ARCHITECTURE: MRL + MNRL
# 1. Loss Function: MultipleNegativesRankingLoss (MNRL)
#    We use in-batch negatives. Every other sample in the batch (size 64) 
#    acts as a negative for the current anchor, providing a strong training signal 
#
# 2. Matryoshka Adaptation
#    The model is forced to learn a hierarchical representation where the 
#    first 64 dimensions carry the vast majority of the semantic signal.

train_matryoshka(pair_df, cfg)

🚀 Loading Backbone: BAAI/bge-base-en-v1.5
⚙️  Config: Batch=64 | Epochs=5 | LR=2e-05
📏 Max Seq Length: 256


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--> Filtering data for Training...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

#### Load Baseline Results (from Notebook 02)

In [2]:
baseline_df = pd.read_csv("../results/baseline_metrics_per_dim.csv")
print(f"✅ Loaded Baseline Results: {len(baseline_df)} rows")

✅ Loaded Baseline Results: 20 rows


### Encoding With Matryoshka Model

In [7]:
matryoshka_path = str(Path(cfg["paths"]["matryoshka_dir"]) / "us")
print(f"🚀 Evaluating Matryoshka Finetuned Model: {matryoshka_path}")
encode_systemA(pair_df, cfg, model_override=matryoshka_path)

🚀 Evaluating Matryoshka Finetuned Model: ..\artifacts\matryoshka_models\us
🚀 Encoding with System A model: ..\artifacts\matryoshka_models\us


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

📏 Max Seq Length: 256
    -> Preparing Product List...
    -> Encoding 164001 Products...


Batches:   0%|          | 0/5126 [00:00<?, ?it/s]

    -> Preparing Query List...
    👉 Applying BGE instruction prefix: 'Represent this sentence for searching relevant passages: '
    -> Encoding 8908 Queries...


Batches:   0%|          | 0/279 [00:00<?, ?it/s]

✅ System A Encoding Complete.


### Loading Sparse Artifacts

In [3]:
print("⚡ Loading SPLADE Matrix...")
splade_data = torch.load("../artifacts/systemA/splade_data.pt", map_location="cpu")

splade_loaded = SPLADEFast(
    cfg["sparse"]["splade_model"], 
    batch_size=cfg["sparse"]["batch_size"]
)

splade_loaded.doc_matrix = splade_data["doc_matrix"]
splade_loaded.pid_list   = splade_data["pid_list"]

# 🔴 CRITICAL: device alignment
splade_loaded.doc_matrix = splade_loaded.doc_matrix.to(splade_loaded.device)

# Safety
assert splade_loaded.doc_matrix.shape[0] == len(splade_loaded.pid_list)

print("⚡ Loading BM25 Index...")
with open("../artifacts/systemA/bm25_retriever.pkl", "rb") as f:
    bm25_loaded = pickle.load(f)

splade = splade_loaded
bm25 = bm25_loaded

⚡ Loading SPLADE Matrix...


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: naver/splade-cocondenser-ensembledistil
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⚡ Loading BM25 Index...


## Running Experiment Loop

In [4]:
results = []
dims = cfg["matryoshka"]["dims"]
strategies = {
    "Dense Only": ["dense"],
    "Dense + BM25": ["dense", "bm25"],
    "Dense + SPLADE": ["dense", "splade"],
    "Dense + BM25 + SPLADE": ["dense", "bm25", "splade"]
}

max_len = cfg["matryoshka"]["max_seq_length"]

for strat, sources in strategies.items():
    print(f"\n🔵 Strategy: {strat}")
    cfg["retrieval"]["sources"] = sources
    
    for dim in dims:
        df, qps = build_candidates(
            cfg, split="test", override_dim=dim,
            prebuilt_splade=splade if "splade" in sources else None,
            prebuilt_bm25=bm25 if "bm25" in sources else None
        )
        
        # Pass q_rels here!
        # Compute them separately
        recall_results = compute_recall_metrics(df, q_rels)
        ndcg_results = compute_ndcg_metrics(df, q_rels)
        
        # Merge them into one final dictionary
        final_metrics = {**recall_results, **ndcg_results}
        
        results.append({
            "Model": "Matryoska",
            "Strategy": strat,
            "Dimension": dim,
            "Max Length": max_len,
            "Recall@50": final_metrics["Recall@50"],
            "Recall@100": final_metrics["Recall@100"],
            "Recall@200": final_metrics["Recall@200"],
            "nDCG@10": final_metrics["nDCG@10"],
            "nDCG@20": final_metrics["nDCG@20"],
            "nDCG@50": final_metrics["nDCG@50"],
            "QPS": round(qps, 2)
        })


🔵 Strategy: Dense Only
⚡ Using FAISS (Dim=768)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=512)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=256)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=128)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=64)
    -> Measuring QPS for 8908 queries...

🔵 Strategy: Dense + BM25
⚡ Using FAISS (Dim=768)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=512)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=256)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=128)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=64)
    -> Measuring QPS for 8908 queries...

🔵 Strategy: Dense + SPLADE
⚡ Using FAISS (Dim=768)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=512)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=256)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=128)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS 

In [9]:
# We compare the Recall@200 of the Fine-Tuned model against the Baseline
# specifically at the 64-dimension truncation.
# Success is defined as: Fine-Tuned@64 > Baseline@64.

In [10]:
# Save Matryoshka results
matryoshka_df = pd.DataFrame(results)
matryoshka_df.to_csv("../results/matryoshka_metrics_per_dim.csv", index=False)
print("✅ Saved Matryoshka metrics to ../results/matryoshka_metrics_per_dim.csv")

✅ Saved Matryoshka metrics to ../results/matryoshka_metrics_per_dim.csv


### Comparing results

In [11]:
# Filter for the primary metrics and keys
cols_to_keep = ['Strategy', 'Dimension', 'Recall@200', 'nDCG@20']
baseline_sub = baseline_df[cols_to_keep]
matryoshka_sub = matryoshka_df[cols_to_keep]

# Merge to compare side-by-side
comparison = pd.merge(
    baseline_sub, 
    matryoshka_sub, 
    on=['Strategy', 'Dimension'], 
    suffixes=('_Base', '_Matryoshka')
)

# Calculate Lifts
comparison['Recall_Lift (%)'] = (
    (comparison['Recall@200_Matryoshka'] - comparison['Recall@200_Base']) / comparison['Recall@200_Base'] * 100
)
comparison['nDCG_Lift (%)'] = (
    (comparison['nDCG@20_Matryoshka'] - comparison['nDCG@20_Base']) / comparison['nDCG@20_Base'] * 100
)

# Display the 64-dimension results (The Matryoshka target)
print("📊 Performance Comparison at 64 Dimensions:")
display_cols = [
    'Strategy', 'Recall@200_Base', 'Recall@200_Matryoshka', 
    'Recall_Lift (%)', 'nDCG@20_Base', 'nDCG@20_Matryoshka', 'nDCG_Lift (%)'
]
display(comparison[comparison['Dimension'] == 64][display_cols].round(2))

📊 Performance Comparison at 64 Dimensions:


,Strategy,Recall@200_Base,Recall@200_Matryoshka,Recall_Lift (%),nDCG@20_Base,nDCG@20_Matryoshka,nDCG_Lift (%)
4,Dense Only,0.43,0.74,73.14,0.25,0.41,63.29
9,Dense + BM25,0.48,0.79,63.61,0.30,0.45,50.26
14,Dense + SPLADE,0.57,0.81,41.16,0.32,0.48,48.75
19,Dense + BM25 + SPLADE,0.65,0.81,24.58,0.35,0.51,45.03


#### The results shows that the finetuned model (Dense @64) was successful at:
#### - truncating the baseline vectors to 64 dimensions.
#### - outperforming the baseline model (Dense @64) Recall@200: 0.74 vs 0.43. (which is basically the same results as baseline@768)
#### The use of Multiple Negatives Ranking Loss (MNRL) was the key of these performances.

### --------------------------------------------------------------------------------------

### For the next step, the logical choice is to stick with Dense+BM25+SPLADE @64 for its performance and acceptable latency.

In [12]:
matryoshka_df

,Model,Strategy,Dimension,Max Length,Recall@50,Recall@100,Recall@200,nDCG@10,nDCG@20,nDCG@50,QPS
0,Matryoska,Dense Only,768,256,0.595176,0.697317,0.776825,0.415088,0.450963,0.510084,43.56
1,Matryoska,Dense Only,512,256,0.591918,0.694383,0.774114,0.411659,0.447071,0.506135,69.28
2,Matryoska,Dense Only,256,256,0.582993,0.686814,0.767909,0.404076,0.439153,0.497337,140.64
3,Matryoska,Dense Only,128,256,0.570365,0.673672,0.757911,0.392224,0.426587,0.485075,267.48
4,Matryoska,Dense Only,64,256,0.548809,0.653159,0.739216,0.373782,0.406898,0.465216,456.29
5,Matryoska,Dense + BM25,768,256,0.623268,0.731759,0.808917,0.436319,0.475208,0.536103,37.43
6,Matryoska,Dense + BM25,512,256,0.621359,0.730568,0.807930,0.434516,0.473108,0.534197,50.87
7,Matryoska,Dense + BM25,256,256,0.617319,0.727530,0.805441,0.431170,0.469188,0.530107,83.47
8,Matryoska,Dense + BM25,128,256,0.608839,0.721051,0.799677,0.424575,0.461837,0.522981,119.19
9,Matryoska,Dense + BM25,64,256,0.594626,0.710708,0.791865,0.414569,0.450534,0.510950,186.69


In [13]:
baseline_df

,Model,Strategy,Dimension,Max Length,Recall@50,Recall@100,Recall@200,nDCG@10,nDCG@20,nDCG@50,QPS
0,Baseline,Dense Only,768,256,0.569735,0.662507,0.739768,0.433009,0.463072,0.515612,36.19
1,Baseline,Dense Only,512,256,0.555104,0.647921,0.724900,0.422693,0.451474,0.503189,56.52
2,Baseline,Dense Only,256,256,0.511757,0.601916,0.681114,0.395021,0.420819,0.469609,108.34
3,Baseline,Dense Only,128,256,0.430857,0.512219,0.588258,0.338874,0.360226,0.403421,204.05
4,Baseline,Dense Only,64,256,0.290365,0.357380,0.426956,0.235852,0.249192,0.281322,329.28
5,Baseline,Dense + BM25,768,256,0.586373,0.679313,0.756639,0.447023,0.478648,0.530854,28.47
6,Baseline,Dense + BM25,512,256,0.573561,0.667018,0.744027,0.439004,0.469240,0.520690,41.81
7,Baseline,Dense + BM25,256,256,0.535216,0.628261,0.708182,0.419462,0.445571,0.494243,68.66
8,Baseline,Dense + BM25,128,256,0.461280,0.547308,0.630038,0.377115,0.396357,0.439051,101.94
9,Baseline,Dense + BM25,64,256,0.326091,0.401858,0.483983,0.291657,0.299827,0.332209,144.04
